In [1]:
# Import Pandas for data handling
import pandas as pd

# Import NumPy for numerical operations
import numpy as np

# Import train-test split function
from sklearn.model_selection import train_test_split

# Import Pipeline
from sklearn.pipeline import Pipeline

# Import ColumnTransformer
from sklearn.compose import ColumnTransformer

# Import missing value handler
from sklearn.impute import SimpleImputer

# Import StandardScaler
from sklearn.preprocessing import StandardScaler

# Import OneHotEncoder
from sklearn.preprocessing import OneHotEncoder

# Import Joblib for saving the pipeline
import joblib

In [2]:
# Load the CKD dataset
df = pd.read_csv("kidney_disease.csv")

# Display first five records
print(df.head())

   id   age    bp     sg   al   su     rbc        pc         pcc          ba  \
0   0  48.0  80.0  1.020  1.0  0.0     NaN    normal  notpresent  notpresent   
1   1   7.0  50.0  1.020  4.0  0.0     NaN    normal  notpresent  notpresent   
2   2  62.0  80.0  1.010  2.0  3.0  normal    normal  notpresent  notpresent   
3   3  48.0  70.0  1.005  4.0  0.0  normal  abnormal     present  notpresent   
4   4  51.0  80.0  1.010  2.0  0.0  normal    normal  notpresent  notpresent   

   ...  pcv    wc   rc  htn   dm  cad appet   pe  ane classification  
0  ...   44  7800  5.2  yes  yes   no  good   no   no            ckd  
1  ...   38  6000  NaN   no   no   no  good   no   no            ckd  
2  ...   31  7500  NaN   no  yes   no  poor   no  yes            ckd  
3  ...   32  6700  3.9  yes   no   no  poor  yes  yes            ckd  
4  ...   35  7300  4.6   no   no   no  good   no   no            ckd  

[5 rows x 26 columns]


In [3]:
# Display dataset information
print(df.info())

# Display missing values
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 26 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              400 non-null    int64  
 1   age             391 non-null    float64
 2   bp              388 non-null    float64
 3   sg              353 non-null    float64
 4   al              354 non-null    float64
 5   su              351 non-null    float64
 6   rbc             248 non-null    object 
 7   pc              335 non-null    object 
 8   pcc             396 non-null    object 
 9   ba              396 non-null    object 
 10  bgr             356 non-null    float64
 11  bu              381 non-null    float64
 12  sc              383 non-null    float64
 13  sod             313 non-null    float64
 14  pot             312 non-null    float64
 15  hemo            348 non-null    float64
 16  pcv             330 non-null    object 
 17  wc              295 non-null    obj

In [4]:
# Separate input features
X = df.drop("classification", axis=1)

# Separate target variable
y = df["classification"]

print("Feature Shape:", X.shape)
print("Target Shape:", y.shape)

Feature Shape: (400, 25)
Target Shape: (400,)


In [5]:
# Identify numerical columns
numerical_features = X.select_dtypes(include=['int64','float64']).columns

# Identify categorical columns
categorical_features = X.select_dtypes(include=['object']).columns

print("Numerical Features")
print(numerical_features)

print()

print("Categorical Features")
print(categorical_features)

Numerical Features
Index(['id', 'age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot',
       'hemo'],
      dtype='object')

Categorical Features
Index(['rbc', 'pc', 'pcc', 'ba', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad',
       'appet', 'pe', 'ane'],
      dtype='object')


In [6]:
# Numerical preprocessing pipeline

numerical_pipeline = Pipeline(

    steps=[

        ('imputer', SimpleImputer(strategy='median')),

        ('scaler', StandardScaler())

    ]

)

In [7]:
# Categorical preprocessing pipeline

categorical_pipeline = Pipeline(

    steps=[

        ('imputer', SimpleImputer(strategy='most_frequent')),

        ('encoder', OneHotEncoder(handle_unknown='ignore'))

    ]

)

In [8]:
# Combine numerical and categorical pipelines

preprocessor = ColumnTransformer(

    transformers=[

        ('num', numerical_pipeline, numerical_features),

        ('cat', categorical_pipeline, categorical_features)

    ]

)

print(preprocessor)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 Index(['id', 'age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot',
       'hemo'],
      dtype='object')),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 Index(['rbc', 'pc', 'pcc', 'ba', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad',
       'appet', 'pe', 'ane'],
      dtype='object'))])


In [9]:
# Split into training and testing datasets

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20,random_state=42, stratify=y)

print("Training Samples:", X_train.shape)

print("Testing Samples:", X_test.shape)

Training Samples: (320, 25)
Testing Samples: (80, 25)


In [10]:
# Fit preprocessing pipeline

preprocessor.fit(X_train)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 Index(['id', 'age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot',
       'hemo'],
      dtype='object')),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 Index(['rbc', 'pc', 'pcc', 'ba', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad',
       'appet', 'pe', 'ane'],
      dtype='object'))])

In [11]:
# Transform training dataset

X_train_processed = preprocessor.transform(X_train)

print(X_train_processed.shape)

(320, 204)


In [12]:
# Transform testing dataset

X_test_processed = preprocessor.transform(X_test)

print(X_test_processed.shape)

(80, 204)


In [13]:
# Convert sparse matrix to dense array (for viewing only)

X_train_dense = X_train_processed.toarray()

print(X_train_dense[:5])

[[ 1.56272731  0.46987456 -1.31317923 ...  0.          1.
   0.        ]
 [-1.2206089   1.4862235  -0.51428692 ...  1.          1.
   0.        ]
 [-0.61927083  1.12751211  1.08349769 ...  1.          1.
   0.        ]
 [ 1.48541242 -1.38346763 -1.31317923 ...  0.          1.
   0.        ]
 [ 1.15897175 -1.68239379  0.28460538 ...  0.          1.
   0.        ]]


In [14]:
print(X_train_processed[:5])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 125 stored elements and shape (5, 204)>
  Coords	Values
  (0, 0)	1.5627273102791253
  (0, 1)	0.4698745569936886
  (0, 2)	-1.3131792261105526
  (0, 3)	0.41229783751625504
  (0, 4)	-0.6848607259475687
  (0, 5)	-0.3947766168272631
  (0, 6)	-0.41670827082134704
  (0, 7)	-0.6728761686572089
  (0, 8)	-0.37851618737515935
  (0, 9)	0.18517575622768412
  (0, 10)	-0.34693192591261146
  (0, 11)	0.9661756571508621
  (0, 13)	1.0
  (0, 15)	1.0
  (0, 16)	1.0
  (0, 18)	1.0
  (0, 58)	1.0
  (0, 112)	1.0
  (0, 170)	1.0
  (0, 188)	1.0
  (0, 193)	1.0
  (0, 196)	1.0
  (0, 198)	1.0
  (0, 200)	1.0
  (0, 202)	1.0
  :	:
  (4, 0)	1.1589717487695044
  (4, 1)	-1.6823937915420142
  (4, 2)	0.28460538360570914
  (4, 3)	0.41229783751625504
  (4, 4)	-0.6848607259475687
  (4, 5)	-0.3947766168272631
  (4, 6)	-0.594250168422834
  (4, 7)	-0.1883094672896175
  (4, 8)	-0.3564291742533412
  (4, 9)	0.6579649210643198
  (4, 10)	-0.1872853681156087
  (4, 11)	1.8709712

In [15]:
# Save pipeline

joblib.dump(preprocessor, "preprocessing_pipeline.pkl")

print("Pipeline saved successfully.")

Pipeline saved successfully.


In [16]:
# Load saved pipeline

loaded_pipeline = joblib.load("preprocessing_pipeline.pkl")

print(loaded_pipeline)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 Index(['id', 'age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot',
       'hemo'],
      dtype='object')),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 Index(['rbc', 'pc', 'pcc', 'ba', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad',
       'appet', 'pe', 'ane'],
      dtype='object'))])


In [17]:
# Transform test dataset using loaded pipeline

new_test = loaded_pipeline.transform(X_test)

print(new_test.shape)

(80, 204)


In [18]:
# Convert sparse matrices to dense arrays
X_train_df = pd.DataFrame(X_train_processed.toarray())
X_test_df = pd.DataFrame(X_test_processed.toarray())

# Save processed datasets
X_train_df.to_csv("X_train_processed.csv", index=False)
X_test_df.to_csv("X_test_processed.csv", index=False)

# Save target variables
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("Processed datasets saved successfully.")

Processed datasets saved successfully.
